# Face Bounding-Box Detection — YOLOv8 vs YOLO11

Compares two YOLO-based face detectors across an easy and a hard image.

| Detector | Model | Landmarks | Notes |
|---|---|---|---|
| `YoloFaceV8Detector` | YOLOv8-Face (`arnabdhar`) | none | Detection-only; no keypoint head |
| `YoloFace11Detector` | YOLO11-pose WIDERFace (`zjykzj`) | 5 pts | More robust; includes facial keypoints |

Both are loaded with `conf=0.7` to keep only high-confidence detections.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from exordium import FIXTURE_DIR
from exordium.video.core.detection import add_detections_to_frame
from exordium.video.core.io import image_to_np

multispeaker_path = FIXTURE_DIR / "image" / "multispeaker.png"  # easy: 3 frontal faces
emma_path = FIXTURE_DIR / "image" / "emma.jpg"  # hard: extreme head poses

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(image_to_np(multispeaker_path))
axes[0].set_title("multispeaker.png — easy (3 frontal faces)")
axes[0].axis("off")
axes[1].imshow(image_to_np(emma_path))
axes[1].set_title("emma.jpg — hard (extreme head poses)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Load detectors

In [ ]:
from exordium.video.face.detector.yolo import YoloFaceV8Detector
from exordium.video.face.detector.yolo11 import YoloFace11Detector

yolo_v8 = YoloFaceV8Detector(device_id=None, conf=0.7)
yolo_11 = YoloFace11Detector(device_id=None, conf=0.7)

## Easy image — `multispeaker.png` (3 frontal faces)

Both detectors are expected to find all three faces at conf=0.7.

In [ ]:
# detect_image_path(path) — one liner: loads image, stores path as source for crop()
ms_v8 = yolo_v8.detect_image_path(multispeaker_path)
ms_11 = yolo_11.detect_image_path(multispeaker_path)

for label, dets in [("YOLOv8", ms_v8), ("YOLO11", ms_11)]:
    print(f"{label}  —  {len(dets)} face(s)")
    for i, det in enumerate(dets):
        print(f"  face {i}  score={det.score:.3f}  bb={[int(v) for v in det.bb_xyxy]}")

In [ ]:
def to_rgb(dets):
    """Overlay BBs on the source image and return RGB numpy for matplotlib."""
    return cv2.cvtColor(add_detections_to_frame(dets, frame=None), cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(to_rgb(ms_v8))
axes[0].set_title(f"YOLOv8 — {len(ms_v8)} face(s)  (detection-only)", fontsize=11)
axes[0].axis("off")
axes[1].imshow(to_rgb(ms_11))
axes[1].set_title(f"YOLO11 — {len(ms_11)} face(s)  (detection + 5-pt keypoints)", fontsize=11)
axes[1].axis("off")
plt.suptitle("Easy image — frontal faces (multispeaker.png, conf=0.7)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# det.crop(square=True) → (3, H', W') uint8 RGB tensor — one liner
crops = [det.crop(square=True, extra_space=1.3) for det in ms_11]

n = len(crops)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]
for ax, crop, det in zip(axes, crops, ms_11):
    ax.imshow(crop.permute(1, 2, 0).numpy())
    ax.set_title(f"score={det.score:.3f}\n{crop.shape[1]}×{crop.shape[2]} px", fontsize=9)
    ax.axis("off")
plt.suptitle("YOLO11 face crops — easy image (square=True, extra_space=1.3)", fontsize=11)
plt.tight_layout()
plt.show()

## Hard image — `emma.jpg` (extreme head poses)

`emma.jpg` contains 3 faces with large yaw / pitch / roll variation.
YOLO11 (trained on WIDERFace with pose supervision) handles non-frontal faces better.

In [ ]:
em_v8 = yolo_v8.detect_image_path(emma_path)
em_11 = yolo_11.detect_image_path(emma_path)

for label, dets in [("YOLOv8", em_v8), ("YOLO11", em_11)]:
    print(f"{label}  —  {len(dets)} face(s)")
    for i, det in enumerate(dets):
        print(f"  face {i}  score={det.score:.3f}  bb={[int(v) for v in det.bb_xyxy]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(to_rgb(em_v8))
axes[0].set_title(f"YOLOv8 — {len(em_v8)} face(s)", fontsize=11)
axes[0].axis("off")
axes[1].imshow(to_rgb(em_11))
axes[1].set_title(f"YOLO11 — {len(em_11)} face(s)", fontsize=11)
axes[1].axis("off")
plt.suptitle("Hard image — extreme head poses (emma.jpg, conf=0.7)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
crops_em = [det.crop(square=True, extra_space=1.3) for det in em_11]

n = len(crops_em)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]
for ax, crop, det in zip(axes, crops_em, em_11):
    ax.imshow(crop.permute(1, 2, 0).numpy())
    ax.set_title(f"score={det.score:.3f}\n{crop.shape[1]}×{crop.shape[2]} px", fontsize=9)
    ax.axis("off")
plt.suptitle("YOLO11 face crops — hard image (square=True, extra_space=1.3)", fontsize=11)
plt.tight_layout()
plt.show()